In [2]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/cs-training.csv", index_col=0)
df = df.rename(columns={
    "SeriousDlqin2yrs": "default_flag",
    "RevolvingUtilizationOfUnsecuredLines": "utilization",
    "NumberOfTime30-59DaysPastDueNotWorse": "late_30_59",
    "NumberOfTimes90DaysLate": "late_90",
    "NumberOfTime60-89DaysPastDueNotWorse": "late_60_89",
    "MonthlyIncome": "income",
    "DebtRatio": "debt_ratio",
})

conn = sqlite3.connect(":memory:")
df.to_sql("loans", conn, index=False)

def q(sql):
    return pd.read_sql(sql, conn)

In [3]:
# 1. first 5 rows
q("SELECT * FROM loans LIMIT 5")

# 2. how many borrowers, how many defaults, default rate
q("""
SELECT COUNT(*) AS borrowers,
       SUM(default_flag) AS defaults,
       ROUND(AVG(default_flag), 4) AS default_rate
FROM loans
""")

# 3. filter: over-limit borrowers
q("SELECT COUNT(*) FROM loans WHERE utilization > 1 AND utilization < 10")

# 4. missing income: count and default rate
q("""
SELECT income IS NULL AS income_missing,
       COUNT(*) AS n,
       ROUND(AVG(default_flag), 4) AS default_rate
FROM loans
GROUP BY income IS NULL
""")

,income_missing,n,default_rate
0,0,120269,0.0695
1,1,29731,0.0561


In [4]:
# 5. default rate by number of 90+ day late payments
q("""
SELECT late_90 AS times_90_late,
       COUNT(*) AS n,
       ROUND(AVG(default_flag), 3) AS default_rate
FROM loans
WHERE late_90 < 90
GROUP BY late_90
ORDER BY late_90
LIMIT 8
""")

# 6. age bands with CASE
q("""
SELECT CASE WHEN age < 30 THEN 'under 30'
            WHEN age < 50 THEN '30-49'
            WHEN age < 65 THEN '50-64'
            ELSE '65+' END AS age_band,
       COUNT(*) AS n,
       ROUND(AVG(default_flag), 3) AS default_rate
FROM loans
WHERE age >= 18
GROUP BY age_band
ORDER BY default_rate DESC
""")

# 7. only groups with enough borrowers (HAVING)
q("""
SELECT late_30_59, COUNT(*) AS n, ROUND(AVG(default_flag), 3) AS default_rate
FROM loans
WHERE late_30_59 < 90
GROUP BY late_30_59
HAVING COUNT(*) >= 100
""")

# 8. subquery: borrowers with income above the average
q("""
SELECT COUNT(*) FROM loans
WHERE income > (SELECT AVG(income) FROM loans)
""")

,COUNT(*)
0,44146
